In [9]:
import numpy as np
from tqdm import tqdm
from embedder import Embedder
embedder = Embedder()

In [10]:
query = "How does approximate nearest neighbor search work?"
v_query = embedder.encode(query)

print(v_query.shape)
print(v_query[0])


(384,)
-0.02058203437252893


In [11]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [12]:
texts = [doc["content"] + " " + doc["filename"] for doc in documents]

In [13]:
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = None
for doc in documents:
    if doc["filename"] == target_file:
        target_doc = doc
        break

v_doc = embedder.encode(target_doc["content"])
cosine_sim = v_query.dot(v_doc)

In [14]:
print(cosine_sim)

0.36107026789538205


In [ ]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)


In [17]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [
        chunk["content"] + " " + chunk["filename"]
        for chunk in chunks[i:i + batch_size]
    ]
    batch_vectors = embedder.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [18]:

scores = X.dot(v_query)
print(scores)

[ 3.15187705e-01  2.01479593e-01  5.90559560e-02 -7.67661858e-02
  1.18452480e-01 -1.41697042e-01 -2.81406552e-02 -4.65669225e-02
 -2.06994704e-02 -6.07744087e-02  2.13273853e-01  8.87601799e-02
 -1.97269351e-02  3.11630016e-01  5.51034674e-01  2.04008048e-01
  2.12515842e-01  1.93649180e-01  2.51961293e-01  1.31078643e-01
  2.59120579e-01  7.63816008e-02  9.59193707e-02  9.81472975e-03
 -3.59106882e-02  1.01211577e-02  1.10786937e-01 -9.90259208e-02
 -3.71170151e-02  7.59057570e-02 -3.35340540e-02  8.86841309e-03
  1.02636405e-01  6.89614876e-02  1.29408856e-01  2.57709091e-01
  3.23680614e-01  1.06350075e-01  5.61891367e-02  2.34017457e-01
  1.97954387e-01  9.64296290e-02  1.93709917e-01  2.16719271e-01
  3.48340456e-01  5.10906092e-02  2.05212837e-01  1.05416170e-01
 -3.25432514e-02  4.94665548e-02  2.38574865e-01 -3.44207108e-02
  1.82165438e-01  2.77929622e-02 -2.26803817e-03  2.91897529e-01
  4.39332572e-02  2.23759709e-01  1.63523594e-01  9.16125938e-02
 -8.39985941e-02  7.88591

In [20]:
best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]
print(best_idx)
print(best_chunk['filename'])

94
02-vector-search/lessons/07-sqlitesearch-vector.md


In [23]:
import minsearch

# Index chunks with vector search
index = minsearch.VectorSearch(
    keyword_fields=["filename"],
    numeric_fields=["start"],
)

index.fit(X, chunks)

query_q4 = "What metric do we use to evaluate a search engine?"
v_q4 = embedder.encode(query_q4)

results_vector = index.search(v_q4, num_results=5)


print(results_vector[0]['filename'])


04-evaluation/lessons/05-search-metrics.md


In [25]:
text_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename", "start"]
)
text_index.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"
v_q5 = embedder.encode(query_q5)

vector_results_q5 = index.search(v_q5, num_results=5)
text_results_q5 = text_index.search(query_q5, num_results=5)

vector_files_q5 = {r["filename"] for r in vector_results_q5}
text_files_q5 = {r["filename"] for r in text_results_q5}

only_vector = vector_files_q5 - text_files_q5


print(f"Vector results: {vector_files_q5}")
print(f"Text results: {text_files_q5}")
print(f"In vector but NOT in text: {only_vector}")

Vector results: {'02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md'}
Text results: {'02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md'}
In vector but NOT in text: {'02-vector-search/lessons/08-pgvector.md'}


In [27]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start", 0))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query_q6 = "How do I give the model access to tools?"
v_q6 = embedder.encode(query_q6)

vector_results_q6 = index.search(v_q6, num_results=10)
text_results_q6 = text_index.search(query_q6, num_results=10)

hybrid_results = rrf([vector_results_q6, text_results_q6])

print(hybrid_results[0]['filename'])


01-agentic-rag/lessons/13-function-calling.md
